# 영어 튜터 Agent 를 한 단계씩 · Chain → Router → Agent

강의 오후 ① (Academy 모듈 1 · L4 Chain · L5 Router · L6 Agent)

| 단계 | 그래프 | 보여 주는 것 |
| --- | --- | --- |
| ① Chain | START → tutor → END | 노드 하나가 LLM 을 부른다. 도구 요청은 오지만, 실행할 자리가 없다 |
| ② Router | tutor ⇢ tools → END | 도구냐 답이냐를 모델이 고른다. 도구는 실행되지만 결과를 모델이 못 본다 |
| ③ Agent | tutor ⇢ tools → tutor | 선 하나만 바꾼다. 결과가 모델에게 돌아가며 루프가 된다 |

이번에는 LLM 을 그래프 안에 넣어 보겠습니다. Academy 모듈 1 의 Chain, Router, Agent 세 단계를 그대로 따라갑니다.

같은 영어 튜터 그래프를 한 단계씩 키워 가면서, 단계마다 무엇이 바뀌는지, 그리고 무엇이 아직 모자란지를 보겠습니다.

모델은 실습과 같은 OpenRouter 의 `qwen/qwen3-32b` 를 씁니다. 접속 정보는 `lab/.env` 에서 읽습니다. 첫 칸을 실행하면 접속 정보가 찍히는데, 키가 "있음" 으로 나오는지만 확인하고 넘어갑니다.

In [ ]:
import "npm:@langchain/core@1.2.14";
import { Annotation, StateGraph, START, END } from "npm:@langchain/langgraph@1.4.18";

// 접속 정보: 실습과 같은 lab/.env 를 읽는다 (OpenRouter · qwen/qwen3-32b). 키는 화면에 찍지 않는다
const ENV: Record<string, string> = {};
try {
  for (const line of Deno.readTextFileSync("../lab/.env").split("\n")) {
    const m = line.match(/^\s*([A-Z_]+)\s*=\s*(.*)\s*$/);
    if (m) ENV[m[1]] = m[2];
  }
} catch { /* lab/.env 가 없으면 아래 기본값 */ }
const BASE_URL = ENV.LLM_BASE_URL ?? "https://openrouter.ai/api/v1";
const API_KEY  = ENV.LLM_API_KEY  ?? "";
const MODEL    = ENV.LLM_MODEL    ?? "qwen/qwen3-32b";
const OPTIONS  = { temperature: 0 };   // 시연이 매번 같게
console.log({ BASE_URL, MODEL, 키: API_KEY ? "있음" : "없음 — lab/.env 에 LLM_API_KEY 를 넣는다" });

// LangSmith (강사 시연용): demo/.env 에 키가 있으면 실행 기록(trace)이 LangSmith 에 남는다. 실습은 lab/.env 를 써서 남지 않는다
import { traceable } from "npm:langsmith@0.11.1/traceable";
try {
  for (const line of Deno.readTextFileSync(".env").split("\n")) {
    const m = line.match(/^\s*(LANGSMITH_[A-Z_]+)\s*=\s*(?:"([^"]*)"|([^#\s]*))/);   // "따옴표 값" 은 그대로, 아니면 # 주석 앞까지
    const value = m?.[2] ?? m?.[3];
    if (m && value) Deno.env.set(m[1], value);
  }
} catch { /* demo/.env 가 없으면 기록하지 않는다 */ }
if (!Deno.env.get("LANGSMITH_API_KEY")) Deno.env.set("LANGSMITH_TRACING", "false");   // 키가 없으면 끈다
// 모델을 부르는 함수를 감싸면 trace 에 "모델 호출" 한 칸으로 남는다. 기록을 안 켜면 그냥 원래 함수로 돈다
const asLLM = <F extends (...a: any[]) => Promise<any>>(fn: F, name: string) => traceable(fn, {
  name, run_type: "llm",
  processInputs: (i: any) => ({ messages: i.args ? i.args[0] : i.input }),
  processOutputs: (o: any) => ({ choices: [{ message: typeof o.outputs === "string" ? { role: "assistant", content: o.outputs } : o }] }),
});
console.log({ LangSmith: Deno.env.get("LANGSMITH_TRACING") === "true" && Deno.env.get("LANGSMITH_API_KEY") ? `기록함 · project ${Deno.env.get("LANGSMITH_PROJECT") ?? "default"}` : "기록 안 함 (demo/.env)" });

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

## 준비: 모델 부르기 · 도구 하나

모델을 부르는 함수는 오전에 만든 `chat()` 그대로입니다. fetch 로 `/chat/completions` 를 부르고, 답에서 role, content, tool_calls 세 칸만 꺼내 돌려줍니다.

도구는 하나만 준비했습니다. `lookup_word`, 영어 단어 하나의 뜻과 바꿔 쓸 쉬운 말을 찾아 주는 도구입니다. 실제 사전 API 대신 단어 네 개짜리 작은 사전을 씁니다. 그래서 시연할 때마다 결과가 같습니다.

도구 설명의 모양은 오전에 본 것과 똑같습니다. 이름, 설명, 그리고 입력 모양입니다.

In [ ]:
// 오전에 만든 chat() 그대로 — 다시 보낼 칸(role · content · tool_calls)만 돌려준다
async function callModel(messages: any[], tools?: any[]) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, tools, ...OPTIONS }),
  });
  const { role, content, tool_calls } = (await res.json()).choices[0].message;
  return { role, content: content?.trim() ?? null, tool_calls };
}
const chat = asLLM(callModel, "chat");   // LangSmith 에 모델 호출로 남긴다

// 도구 하나: 단어 뜻과 쉬운 말 찾기 (작은 사전이라 시연이 매번 같다)
const DICT: Record<string, { 뜻: string; 쉬운말: string }> = {
  ubiquitous: { 뜻: "어디에나 있는", 쉬운말: "everywhere" },
  mitigate:   { 뜻: "완화하다, 줄이다", 쉬운말: "reduce" },
  meticulous: { 뜻: "꼼꼼한", 쉬운말: "careful" },
  detrimental:{ 뜻: "해로운", 쉬운말: "harmful" },
};
const tools = [{
  type: "function",
  function: {
    name: "lookup_word",
    description: "영어 단어 하나의 뜻과, 바꿔 쓸 쉬운 말을 사전에서 찾는다",
    parameters: { type: "object", properties: { word: { type: "string", description: "찾을 영어 단어 하나" } }, required: ["word"] },
  },
}];
function lookupWord(word: string) {
  const w = word.toLowerCase().trim();
  return DICT[w] ? { word: w, ...DICT[w] } : { word: w, error: "사전에 없는 단어" };
}

const SYSTEM = { role: "system", content: "너는 IELTS 쓰기를 돕는 영어 튜터다. 단어 뜻은 추측하지 말고 lookup_word 로 확인한다. 한국어로 짧게 답한다." };
const ask = (q: string) => ({ messages: [SYSTEM, { role: "user", content: q }] });
function show(out: any) {
  const last = out.messages.at(-1);
  console.log(`messages ${out.messages.length}개 · 마지막: ${last.role}`);
  console.log(last.tool_calls?.length ? `  도구 요청 ${JSON.stringify(last.tool_calls.map((c: any) => c.function))}` : `  ${last.content}`);
}

## State

State 는 이번엔 칸이 하나입니다. `messages`, 대화 전체입니다.

앞 노트북의 `log` 처럼 이어 붙이는 reducer 를 달았습니다. 노드가 모델의 답을 돌려주면 대화 뒤에 붙습니다. 오전에 `messages.push` 로 직접 하던 일을 이제 LangGraph 가 대신 합니다.

In [ ]:
// State: 대화 전체. 노드가 돌려준 메시지는 뒤에 이어 붙는다
const State = Annotation.Root({
  messages: Annotation<any[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
});

## ① Chain — 노드 하나가 LLM 을 부른다

첫 단계는 Chain 입니다. 노드 하나, `tutor` 를 만들고 그 안에서 LLM 을 부릅니다. 도구 목록도 같이 알려 줍니다.

그래프는 START 에서 tutor, tutor 에서 END. 한 줄짜리 그래프입니다.

질문 두 개를 넣어 보겠습니다. 첫 번째는 결론 문단을 어떻게 쓰는지, 도구가 필요 없는 질문입니다. 두 번째는 ubiquitous 의 뜻을 사전에서 찾아 달라는, 도구가 필요한 질문입니다.

In [ ]:
// ① Chain — 노드 하나가 LLM 을 부른다. START → tutor → END
async function tutor(state: typeof State.State) {
  console.log("▶ tutor   LLM 을 부른다");
  return { messages: [await chat(state.messages, tools)] };
}

const chain = new StateGraph(State)
  .addNode("tutor", tutor)
  .addEdge(START, "tutor")
  .addEdge("tutor", END)
  .compile();
await showGraph(chain);

show(await chain.invoke(ask("IELTS 쓰기에서 결론 문단은 어떻게 써?")));
show(await chain.invoke(ask("ubiquitous 뜻을 사전에서 찾아 줘")));   // 도구 요청만 남기고 끝난다 — 실행할 노드가 없다

**돌려 보면** 첫 질문은 바로 답이 나옵니다. 두 번째 질문은 답 대신 **도구 요청만 남기고 끝납니다.**

모델은 `lookup_word` 를 ubiquitous 로 불러 달라고 했지만, 그래프에 도구를 실행할 노드가 없습니다. 요청만 있고 실행이 없습니다. 오전에 "LLM 은 실제로 도구를 실행하지 않는다" 고 했던 바로 그 장면입니다.

## ② Router — 도구 노드와 갈림길을 더한다

그럼 도구를 실행할 노드를 붙여 보겠습니다. `tools` 노드는 마지막 메시지의 tool_calls 를 꺼내 `lookup_word` 를 실행하고, 결과를 `role: "tool"` 메시지로 돌려줍니다. 오전에 직접 구현한 코드와 같습니다.

그리고 tutor 다음에 갈림길을 둡니다. `route` 함수가 마지막 메시지를 보고, 도구 요청이 있으면 tools 로, 없으면 END 로 보냅니다.

여기서 중요한 점은, 이 갈림길을 실제로 고르는 것이 모델이라는 것입니다. 모델이 도구를 원하면 tools 로, 바로 답하면 끝으로 갑니다. 앞에서 "여기서부터를 Agent 라고 부른다" 고 했던 라우터입니다.

tools 가 끝나면 일단 END 로 가게 해 두겠습니다.

In [ ]:
// ② Router — 도구를 실행할 노드와, 도구냐 답이냐를 고르는 갈림길을 더한다. tools → END
function toolsNode(state: typeof State.State) {
  const calls = state.messages.at(-1).tool_calls;
  console.log(`▶ tools   ${calls.map((c: any) => c.function.arguments).join(" · ")}`);
  return { messages: calls.map((c: any) => ({
    role: "tool", tool_call_id: c.id, content: JSON.stringify(lookupWord(JSON.parse(c.function.arguments).word)),
  })) };
}
function route(state: typeof State.State) {
  return state.messages.at(-1).tool_calls?.length ? "tools" : END;   // 모델이 도구를 원하면 tools, 아니면 끝
}

const router = new StateGraph(State)
  .addNode("tutor", tutor)
  .addNode("tools", toolsNode)
  .addEdge(START, "tutor")
  .addConditionalEdges("tutor", route, ["tools", END])
  .addEdge("tools", END)
  .compile();
await showGraph(router);

show(await router.invoke(ask("IELTS 쓰기에서 결론 문단은 어떻게 써?")));   // tools 를 건너뛰고 끝
show(await router.invoke(ask("ubiquitous 뜻을 사전에서 찾아 줘")));            // 사전 결과만 남고 끝 — 모델은 결과를 못 봤다

**돌려 보면** 결론 질문은 tools 를 건너뛰고 바로 끝납니다. 단어 질문은 이번엔 tools 가 실제로 사전을 찾습니다.

그런데 마지막 메시지가 사전 결과, 날것의 JSON 입니다. 학생에게 줄 말이 없습니다. 도구는 돌았지만, **모델이 그 결과를 한 번도 보지 못하고** 그래프가 끝났기 때문입니다.

## ③ Agent — 선 하나만 바꾼다

그러면 어떻게 하면 될까요. 답은 선 하나입니다. tools 에서 END 로 가던 선을, tools 에서 다시 tutor 로 돌려보냅니다. 코드에서 바뀐 곳은 `addEdge` 한 줄뿐입니다.

이제 도구 결과가 모델에게 돌아갑니다. 모델은 결과를 보고 다시 판단합니다. 도구가 더 필요하면 또 부르고, 충분하면 답하고 끝냅니다. 오전에 본 Thought, Action, Observation 의 루프가 그대로 그래프의 선이 됐습니다.

이번엔 단어 두 개를 찾아서, 쉬운 말로 바꾼 영어 문장을 만들어 달라고 해 보겠습니다.

In [ ]:
// ③ Agent — 선 하나만 바꾼다: tools → END 대신 tools → tutor. 결과가 모델에게 돌아가며 루프가 된다
const agent = new StateGraph(State)
  .addNode("tutor", tutor)
  .addNode("tools", toolsNode)
  .addEdge(START, "tutor")
  .addConditionalEdges("tutor", route, ["tools", END])
  .addEdge("tools", "tutor")                                   // ← 여기만 바뀌었다
  .compile();
await showGraph(agent);

show(await agent.invoke(ask("이 문장의 ubiquitous 랑 mitigate 를 사전에서 찾아서, 쉬운 말로 바꾼 영어 문장을 만들어 줘: Smartphones are ubiquitous, and schools try to mitigate their effects.")));

**돌려 보면** tutor 가 ubiquitous 와 mitigate 를 한꺼번에 찾아 달라고 요청하고, tools 가 둘 다 찾은 뒤, 다시 tutor 로 돌아와 이렇게 답합니다.

> Smartphones are everywhere, and schools try to reduce their effects.

사전이 알려 준 쉬운 말 everywhere 와 reduce 가 그대로 쓰였습니다. 오전에 for 루프로 만든 Agent 가 LangGraph 에서는 선 하나입니다.

## 상한 — recursionLimit

마지막으로 하나만 더 보겠습니다. 이 루프는 모델이 "끝" 할 때까지 돕니다. 그럼 모델이 계속 도구를 부르면 어떻게 될까요.

LangGraph 는 그래프가 끝없이 돌지 않게 recursionLimit 이라는 상한을 둡니다. 기본값은 25 단계입니다. 여기서는 일부러 2 로 낮춰서 걸리게 해 보겠습니다.

In [ ]:
// 루프는 모델이 "끝" 할 때까지 돈다. 그래서 상한이 있다 — recursionLimit (기본 25 단계)
try {
  await agent.invoke(ask("meticulous 와 detrimental 을 사전에서 찾아서 예문을 만들어 줘"), { recursionLimit: 2 });
} catch (e) {
  console.log(`${(e as Error).name}: ${(e as Error).message.split("\n")[0]}`);
}

**돌려 보면** tutor 한 번, tools 한 번, 두 단계를 쓰고 다시 tutor 로 가려는 순간 `GraphRecursionError` 로 멈춥니다.

이건 사고를 막는 마지막 안전망입니다. 정상적으로 멈추는 길은 따로 있어야 합니다. 오전에 넣었던 MAX_STEPS 같은 멈춤을 그래프의 어디에 넣으면 될지는, 실습에서 직접 해 보겠습니다.